In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_source (
session_id INT,
zone_id STRING,
vehicle_type STRING,
duration_hours INT,
parking_fee INT,
status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_sessions_source VALUES
(101, 'Z01', 'Car', 3, 150, 'Completed'),
(102, 'Z02', 'Bike', 2, 40, 'Completed'),
(103, 'Z03', 'Car', 5, 250, 'Completed'),
(104, 'Z01', 'SUV', 4, 240, 'Active'),
(105, 'Z04', 'Bike', 1, 20, 'Completed'),
(106, 'Z02', 'Car', 6, 300, 'Completed'),
(107, 'Z03', 'SUV', 2, 120, 'Cancelled'),
(108, 'Z05', 'Car', 3, 150, 'Completed'),
(109, 'Z01', 'Bike', 4, 80, 'Completed'),
(110, 'Z04', 'SUV', 5, 300, 'Active'),
(111, 'Z02', 'Car', 2, 100, 'Completed'),
(112, 'Z05', 'Bike', 3, 60, 'Cancelled'),
(113, 'Z03', 'Car', 7, 350, 'Completed'),
(114, 'Z01', 'SUV', 3, 180, 'Completed'),
(115, 'Z04', 'Bike', 2, 40, 'Active'),
(116, 'Z05', 'Car', 4, 200, 'Completed'),
(117, 'Z02', 'SUV', 6, 360, 'Completed'),
(118, 'Z03', 'Bike', 1, 20, 'Completed'),
(119, 'Z04', 'Car', 5, 250, 'Completed'),
(120, 'Z05', 'SUV', 2, 120, 'Active');

num_affected_rows,num_inserted_rows
20,20


In [0]:
%sql
CREATE OR REPLACE TABLE parking_sessions_work
USING DELTA
AS
SELECT * FROM parking_sessions_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE parking_zones (
zone_id STRING,
zone_name STRING,
city STRING,
capacity INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO parking_zones VALUES
('Z01', 'Central Plaza', 'Hyderabad', 100),
('Z02', 'Metro Parking', 'Bengaluru', 150),
('Z03', 'City Mall', 'Mumbai', 200),
('Z04', 'Airport Parking', 'Delhi', 250),
('Z05', 'Tech Park', 'Pune', 120);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_parking_sessions AS
SELECT *
FROM VALUES
(102, 'Z02', 'Bike', 3, 60, 'Completed'),
(104, 'Z01', 'SUV', 5, 300, 'Completed'),
(121, 'Z03', 'Car', 4, 200, 'Completed'),
(122, 'Z05', 'Bike', 2, 40, 'Active')
AS t(
session_id,
zone_id,
vehicle_type,
duration_hours,
parking_fee,
status
);

Section A — SQL and DataFrames

In [0]:
%sql
-- 1. Display all records from parking_sessions_source. Retrieve only session_id, vehicle_type, and parking_fee.
SELECT session_id, vehicle_type, parking_fee 
FROM parking_sessions_source;

session_id,vehicle_type,parking_fee
101,Car,150
102,Bike,40
103,Car,250
104,SUV,240
105,Bike,20
106,Car,300
107,SUV,120
108,Car,150
109,Bike,80
110,SUV,300


In [0]:
%sql
-- 2. Find all completed parking sessions where the fee is greater than ₹100.
SELECT * 
FROM parking_sessions_source 
WHERE status = 'Completed' AND parking_fee > 100;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed
117,Z02,SUV,6,360,Completed
119,Z04,Car,5,250,Completed


In [0]:
%sql
-- 3. Calculate the total parking fees and number of completed sessions for each vehicle type using GROUP BY.
SELECT 
    vehicle_type, 
    COUNT(*) AS completed_sessions_count, 
    SUM(parking_fee) AS total_fees
FROM parking_sessions_source
WHERE status = 'Completed'
GROUP BY vehicle_type;

vehicle_type,completed_sessions_count,total_fees
Car,8,1750
Bike,4,160
SUV,2,540


In [0]:
# 4. Using PySpark, read parking_sessions_source into a DataFrame, add a column named fee_category (High for fees of ₹200 or more, otherwise Low), and display the result.
from pyspark.sql import functions as F

df = spark.read.table("parking_sessions_source")

df_categorized = df.withColumn(
    "fee_category",
    F.when(F.col("parking_fee") >= 200, "High").otherwise("Low")
)

display(df_categorized)

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status,fee_category
101,Z01,Car,3,150,Completed,Low
102,Z02,Bike,2,40,Completed,Low
103,Z03,Car,5,250,Completed,High
104,Z01,SUV,4,240,Active,High
105,Z04,Bike,1,20,Completed,Low
106,Z02,Car,6,300,Completed,High
107,Z03,SUV,2,120,Cancelled,Low
108,Z05,Car,3,150,Completed,Low
109,Z01,Bike,4,80,Completed,Low
110,Z04,SUV,5,300,Active,High


Section B — Delta Tables and CRUD

In [0]:
%sql
-- 5. Create a new managed Delta table named parking_summary using CTAS, containing only completed sessions.
CREATE OR REPLACE TABLE parking_summary
USING DELTA
AS
SELECT * 
FROM parking_sessions_source
WHERE status = 'Completed';

SELECT * FROM parking_summary;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed


In [0]:
%sql
-- 6. Insert a new record into parking_sessions_work with session ID 123, zone Z01, vehicle type Car, duration 3 hours, fee ₹150, and status Active.
INSERT INTO parking_sessions_work 
VALUES (123, 'Z01', 'Car', 3, 150, 'Active');

SELECT * FROM parking_sessions_work WHERE session_id = 123;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
123,Z01,Car,3,150,Active


In [0]:
%sql
-- 7. Update session 104 to Completed and increase its parking fee to ₹280.
UPDATE parking_sessions_work
SET status = 'Completed', parking_fee = 280
WHERE session_id = 104;

SELECT * FROM parking_sessions_work WHERE session_id = 104;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
104,Z01,SUV,4,280,Completed


In [0]:
%sql
-- 8. Delete all Cancelled sessions from parking_sessions_work. Display the remaining records.
DELETE FROM parking_sessions_work
WHERE status = 'Cancelled';

SELECT * FROM parking_sessions_work;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,2,40,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed


Section C — MERGE and Delta Operations

In [0]:
%sql
-- 9. Display incoming_parking_sessions and identify which records already exist in parking_sessions_work.
SELECT * FROM incoming_parking_sessions;
SELECT i.*
FROM incoming_parking_sessions i
INNER JOIN parking_sessions_work w
ON i.session_id = w.session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
102,Z02,Bike,3,60,Completed
104,Z01,SUV,5,300,Completed


In [0]:
%sql
-- 10. Perform a MERGE operation using session_id to update existing records and insert new records.
MERGE INTO parking_sessions_work AS target
USING incoming_parking_sessions AS source
ON target.session_id = source.session_id
WHEN MATCHED THEN
  UPDATE SET
    target.zone_id = source.zone_id,
    target.vehicle_type = source.vehicle_type,
    target.duration_hours = source.duration_hours,
    target.parking_fee = source.parking_fee,
    target.status = source.status
WHEN NOT MATCHED THEN
  INSERT (session_id, zone_id, vehicle_type, duration_hours, parking_fee, status)
  VALUES (source.session_id, source.zone_id, source.vehicle_type, source.duration_hours, source.parking_fee, source.status);

SELECT * FROM parking_sessions_work ORDER BY session_id;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
102,Z02,Bike,3,60,Completed
103,Z03,Car,5,250,Completed
104,Z01,SUV,5,300,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
110,Z04,SUV,5,300,Active
111,Z02,Car,2,100,Completed


In [0]:
%sql
-- 11. Use DESCRIBE HISTORY to display the transaction history of parking_sessions_work.
DESCRIBE HISTORY parking_sessions_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-06T04:27:42.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2717862303965751),fda86834-c169-40ab-a6b2-29f2965884a6,1106-042057-hkzc70uz-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 9875, p25FileSize -> 2287, numDeletionVectorsRemoved -> 1, minFileSize -> 2287, numAddedFiles -> 1, maxFileSize -> 2287, p75FileSize -> 2287, p50FileSize -> 2287, numAddedBytes -> 2287)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-06T04:27:40.000Z,70541099100131,jersilgm@gmail.com,MERGE,"Map(predicate -> [""(session_id#13563 = session_id#13545)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2717862303965751),fda86834-c169-40ab-a6b2-29f2965884a6,1106-042057-hkzc70uz-v2n,4,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 7597, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 3822, materializeSourceTimeMs -> 305, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1094, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2358)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-06T04:26:39.000Z,70541099100131,jersilgm@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(2717862303965751),3e1c0563-3291-4049-81d7-b632fe67a47e,1106-042057-hkzc70uz-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6092, p25FileSize -> 2278, numDeletionVectorsRemoved -> 1, minFileSize -> 2278, numAddedFiles -> 1, maxFileSize -> 2278, p75FileSize -> 2278, p50FileSize -> 2278, numAddedBytes -> 2278)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T04:26:37.000Z,70541099100131,jersilgm@gmail.com,DELETE,"Map(predicate -> [""(status#12807 = Cancelled)""])",null,List(2717862303965751),3e1c0563-3291-4049-81d7-b632fe67a47e,1106-042057-hkzc70uz-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1008, numDeletionVectorsUpdated -> 1, numDeletedRows -> 2, scanTimeMs -> 580, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 427)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T04:26:26.000Z,70541099100131,jersilgm@gmail.com,UPDATE,"Map(predicate -> [""(session_id#12239 = 104)""])",null,List(2717862303965751),de9568b8-06bc-4555-aa1a-f85ff1c06713,1106-042057-hkzc70uz-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3457, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1630, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1932, rewriteTimeMs -> 1786)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T04:25:13.000Z,70541099100131,jersilgm@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2717862303965751),0d66d7f2-a880-47ca-9780-382ddb42567c,1106-042057-hkzc70uz-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1878)",n

In [0]:
%sql
-- 12. Use Time Travel to retrieve the table version immediately before the MERGE operation. Compare session 102 before and after MERGE.
SELECT 'Before MERGE' AS state, * 
FROM parking_sessions_work VERSION AS OF 4
WHERE session_id = 102

UNION ALL

SELECT 'After MERGE' AS state, * 
FROM parking_sessions_work 
WHERE session_id = 102;

state,session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
Before MERGE,102,Z02,Bike,2,40,Completed
After MERGE,102,Z02,Bike,3,60,Completed


Section D — Tables, Views and Optimization

In [0]:
%sql
-- 13. Create a permanent view named completed_parking_view containing only completed sessions from parking_sessions_work.
CREATE OR REPLACE VIEW completed_parking_view AS
SELECT * 
FROM parking_sessions_work
WHERE status = 'Completed';

SELECT * FROM completed_parking_view;

session_id,zone_id,vehicle_type,duration_hours,parking_fee,status
101,Z01,Car,3,150,Completed
103,Z03,Car,5,250,Completed
105,Z04,Bike,1,20,Completed
106,Z02,Car,6,300,Completed
108,Z05,Car,3,150,Completed
109,Z01,Bike,4,80,Completed
111,Z02,Car,2,100,Completed
113,Z03,Car,7,350,Completed
114,Z01,SUV,3,180,Completed
116,Z05,Car,4,200,Completed


In [0]:
%sql
-- 14. Create a temporary view containing parking sessions from zone Z01 and a global temporary view containing sessions with parking fees above ₹200. Query both views.

CREATE OR REPLACE TEMP VIEW z01_parking_view AS
SELECT * 
FROM parking_sessions_work 
WHERE zone_id = 'Z01';

SELECT * FROM z01_parking_view;

CREATE OR REPLACE GLOBAL TEMP VIEW high_fee_parking_view AS
SELECT * 
FROM parking_sessions_work 
WHERE parking_fee > 200;

SELECT * FROM global_temp.high_fee_parking_view;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8790204214358190>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- 14. Create a temporary view containing parking sessions from zone Z01 and a global temporary view containing sessions with parking fees above ₹200. Query both views.\n\nCREATE OR REPLACE TEMP VIEW z01_parking_view AS\nSELECT * \nFROM parking_sessions_work \nWHERE zone_id = 'Z01';\n\nSELECT * FROM z01_parking_view;\n\nCREATE OR REPLACE GLOBAL TEMP VIEW high_fee_parking_view AS\nSELECT * \nFROM parking_sessions_work \nWHERE parking_fee > 200;\n\nSELECT * FROM global_temp.high_fee_parking_view;\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*a

In [0]:
%sql
-- 15. Join parking_sessions_work with parking_zones and display the zone name, city, vehicle type, and parking fee.
SELECT 
    z.zone_name,
    z.city,
    w.vehicle_type,
    w.parking_fee
FROM parking_sessions_work w
INNER JOIN parking_zones z
ON w.zone_id = z.zone_id;

zone_name,city,vehicle_type,parking_fee
Central Plaza,Hyderabad,Car,150
City Mall,Mumbai,Car,250
Airport Parking,Delhi,Bike,20
Metro Parking,Bengaluru,Car,300
Tech Park,Pune,Car,150
Central Plaza,Hyderabad,Bike,80
Airport Parking,Delhi,SUV,300
Metro Parking,Bengaluru,Car,100
City Mall,Mumbai,Car,350
Central Plaza,Hyderabad,SUV,180


In [0]:
%sql
-- 16. Run DESCRIBE DETAIL on parking_sessions_work, execute OPTIMIZE, and compare the file count before and after optimization. Explain why a small dataset might show no significant change.

DESCRIBE DETAIL parking_sessions_work;

OPTIMIZE parking_sessions_work;

DESCRIBE DETAIL parking_sessions_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,4bb6e464-786f-49af-a8e0-cc03ac8dbb25,workspace.default.parking_sessions_work,null,,2026-11-06T04:21:43.373Z,2026-11-06T04:27:42.000Z,List(),List(),1,2287,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
